# BTC Uncertain Volatility Backbone — Analysis Notebook

**Mirrors the SPX pipeline** (`Uncertain Vol - OTM Backbone Fits.ipynb`) but adapted for BTC options.

Key differences from SPX:
- BTC options are **cash-settled in BTC** (quanto-style payoff: max(S_T-K,0)/S_T)
- Uses **BTC-settled pricing formulas** in `option_pricers.py`
- IV sourced from Deribit's `iv` field (computed correctly for BTC settlement)
- **r = 0** for backbone fitting (short-dated BTC options insensitive to rate)
- **08:00 UTC** observation time, tau computed from exact timestamps
- Regimes detected via PELT (not fixed year boundaries)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.interpolate import interp1d
import pickle
import os

import option_pricers as op
import importlib
importlib.reload(op)

TARGET_TAU = 30 / 365
R = 0.0  # r=0 for BTC backbone fitting

## 1. Load Data & Define Regimes

In [ ]:
# Load BTC ATM and OTM data (generated by btc_data_processing.py)
# These files will be created after running the download + processing pipeline

atm_path = './processed_data/btc_1M_ATM.csv'
otm_path = './processed_data/btc_iv1m_k_pc.csv'
fits_path = './processed_data/btc_fits_regime_mse.pkl'

if os.path.exists(atm_path):
    iv1m_df = pd.read_csv(atm_path, parse_dates=['date'])
    iv1m_df['year'] = iv1m_df['date'].dt.year
    print(f"Loaded BTC ATM data: {len(iv1m_df)} observations")
    print(f"Date range: {iv1m_df['date'].min()} to {iv1m_df['date'].max()}")
    print(f"ATM IV stats:\n{iv1m_df['atm_iv_1m'].describe()}")
else:
    print(f"ERROR: {atm_path} not found.")
    print("Run btc_data_download.py and btc_data_processing.py first.")
    
if os.path.exists(otm_path):
    iv1m_k_pc_df = pd.read_csv(otm_path, parse_dates=['date'])
    iv1m_k_pc_df['year'] = iv1m_k_pc_df['date'].dt.year
    print(f"\nLoaded BTC OTM data: {len(iv1m_k_pc_df)} observations")
    print(f"Moneyness levels: {sorted(iv1m_k_pc_df['k'].unique())}")
else:
    print(f"\nWARNING: {otm_path} not found.

## 2. Regime Detection

In [ ]:
# Year-based regimes (tentative, to be confirmed by PELT)
def regime_btc(year):
    if year <= 2020:
        return 'COVID/Recovery'
    elif year == 2021:
        return 'Bull'
    elif year == 2022:
        return 'Bear/FTX'
    else:
        return 'Recovery/ETF'

if 'regime' not in iv1m_df.columns:
    iv1m_df['regime'] = iv1m_df['year'].apply(regime_btc)
    
regime_colors = {
    'COVID/Recovery': '#1f77b4',
    'Bull':            '#ff7f0e',
    'Bear/FTX':        '#d62728',
    'Recovery/ETF':    '#2ca02c',
}

REGIMES = list(regime_colors.keys())

# PELT change-point detection (optional, uncomment when ruptures is installed)
# try:
#     import ruptures as rpt
#     signal = iv1m_df.sort_values('date')['atm_iv_1m'].values
#     algo = rpt.Pelt(custom_cost=rpt.costs.CostL2()).fit(signal)
#     breakpoints = algo.predict(pen=10)
#     print(f"PELT breakpoints: {breakpoints}")
# except ImportError:
#     print("ruptures not installed, using year-based regimes")

print(f"Regime distribution:\n{iv1m_df['regime'].value_counts()}")

## 3. ATM Volatility Backbone (Scatter)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Full scatter
ax = axes[0]
ax.scatter(iv1m_df['spot'], iv1m_df['atm_iv_1m'], s=8, alpha=0.3, c='steelblue')
ax.set_xlabel('BTC Spot Price (USD)')
ax.set_ylabel('1M ATM Implied Volatility')
ax.set_title('BTC Volatility Backbone')
ax.grid(alpha=0.3)

# Plot 2: Color by regime
ax = axes[1]
for reg, g in iv1m_df.groupby('regime'):
    ax.scatter(g['spot'], g['atm_iv_1m'], s=8, alpha=0.35,
               color=regime_colors.get(reg, 'gray'), label=reg)
ax.set_xlabel('BTC Spot Price (USD)')
ax.set_ylabel('1M ATM Implied Volatility')
ax.set_title('BTC Volatility Backbone by Regime')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Desciptive stats by regime
print("\nATM IV stats by regime:")
for reg, g in iv1m_df.groupby('regime'):
    print(f"  {reg:20s}: mean={g['atm_iv_1m'].mean():.4f}  "
          f"std={g['atm_iv_1m'].std():.4f}  "
          f"median={g['atm_iv_1m'].median():.4f}  "
          f"n={len(g)}")

## 4. Volatility Backbone Changes (Delta-Vol)

In [ ]:
iv1m_df = iv1m_df.sort_values('date')
iv1m_df['d_iv'] = iv1m_df['atm_iv_1m'].diff()
iv1m_df['d_log_spot'] = np.log(iv1m_df['spot']).diff()
chg = iv1m_df.dropna()

try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
    chg_sorted = chg.sort_values('d_log_spot')
    smoothed = lowess(chg_sorted['d_iv'], chg_sorted['d_log_spot'], frac=0.2)
    has_lowess = True
except ImportError:
    has_lowess = False

plt.figure(figsize=(10, 6))
plt.scatter(chg['d_log_spot'], chg['d_iv'], s=8, alpha=0.3, label='Daily Changes')

if has_lowess:
    plt.plot(smoothed[:, 0], smoothed[:, 1], color='red', linewidth=2.5, label='LOWESS Fit')

plt.xlabel('Δ log(BTC Spot)')
plt.ylabel('Δ 1M ATM Vol')
plt.title('BTC Volatility Backbone (Changes)')
if has_lowess:
    plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 5. Model Fitting: 3-State Displaced-Diffusion Mixture

Uses BTC-settled pricing formulas (`uncertain_backbone_BTC`). Key changes from SPX:
- `r = 0` (BTC short-dated options insensitive to rate)
- `beta` bounds `(0.05, 2.0)` (allow symmetric/slight-positive skew)
- `sigma_ln` bounds `(0.05, 5.0)` (wider for BTC's higher vol)


In [ ]:
def loss_mse_btc(params, S_data, iv_data, r, T):
    weights = params[0:3]
    betas = params[3:6]
    sigmas_ln = params[6:9]
    weights = np.exp(weights) / np.sum(np.exp(weights))
    S_ref = np.mean(S_data)
    sigmas_n = list(np.array(sigmas_ln) * S_ref)
    if np.any(betas <= 0) or np.any(sigmas_ln <= 0):
        return 1e6
    model_ivs = [op.uncertain_backbone_BTC(S, S, r, T, weights, betas, sigmas_n, sigmas_ln)
                 for S in S_data]
    return np.mean((np.array(model_ivs) - np.array(iv_data))**2)

### 5.1 Fit per Regime

In [ ]:
# Fit or load regime parameters
if os.path.exists(fits_path):
    with open(fits_path, 'rb') as f:
        fits_regime_mse = pickle.load(f)
    print(f"Loaded fits for regimes: {list(fits_regime_mse.keys())}")
    for reg, info in fits_regime_mse.items():
        p = info['params']
        w = np.exp(p[:3]) / np.sum(np.exp(p[:3]))
        print(f"  {reg}:")
        print(f"    Converged: {info['success']},  MSE: {info['fun']:.6f}")
        print(f"    weights  = {w}")
        print(f"    betas    = {p[3:6]}")
        print(f"    sigmas_ln = {np.abs(p[6:9])}")
else:
    fits_regime_mse = {}
    
    for regime_name, g in iv1m_df.groupby('regime'):
        S_data = g['spot'].values
        iv_data = g['atm_iv_1m'].values
    
        init = np.array([
            0.3, 0.3, 0.4,   # weights (pre-softmax)
            1.0, 1.2, 0.8,   # betas (centered around 1 for BTC)
            0.5, 1.0, 2.0    # sigmas_ln (higher for BTC vol levels)
        ])
    
        bounds = [
            (None, None), (None, None), (None, None),  # weights
            (0.05, 2.0), (0.05, 2.0), (0.05, 2.0),     # betas
            (0.05, 5.0), (0.05, 5.0), (0.05, 5.0)      # sigmas_ln
        ]
    
        res = minimize(
            loss_mse_btc, init,
            args=(S_data, iv_data, R, TARGET_TAU),
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': 1000}
        )
    
        fits_regime_mse[regime_name] = {
            'params': res.x,
            'success': res.success,
            'fun': res.fun,
            'niter': res.nit
        }
    
        p = res.x
        w = np.exp(p[:3]) / np.sum(np.exp(p[:3]))
        print(f'{regime_name}: MSE={res.fun:.6f}, converged={res.success}')
        print(f'  weights={w}, betas={p[3:6]}, sigmas_ln={np.abs(p[6:9])}')
    
    with open(fits_path, 'wb') as f:
        pickle.dump(fits_regime_mse, f)
    print(f"\nSaved fits to {fits_path}")

### 5.2 ATM Backbone Fitted Plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for reg in REGIMES:
    if reg not in fits_regime_mse:
        continue
    g = iv1m_df[iv1m_df['regime'] == reg]
    ax.scatter(g['spot'], g['atm_iv_1m'], s=6, alpha=0.3,
               color=regime_colors[reg], label=reg)

    params = fits_regime_mse[reg]['params']
    raw_w = params[:3]
    weights = np.exp(raw_w) / np.sum(np.exp(raw_w))
    betas = params[3:6]
    sigmas_ln = np.abs(params[6:9])
    S_ref = g['spot'].mean()
    sigmas_n = list(np.array(sigmas_ln) * S_ref)

    S_grid = np.linspace(g['spot'].min(), g['spot'].max(), 200)
    fitted = [
        op.uncertain_backbone_BTC(S, S, R, TARGET_TAU, weights, betas, sigmas_n, sigmas_ln)
        for S in S_grid
    ]

    ax.plot(S_grid, fitted, color=regime_colors[reg], linewidth=2.5)

ax.set_xlabel('BTC Spot Price (USD)')
ax.set_ylabel('1M ATM Implied Volatility')
ax.set_title('BTC ATM Backbone: Uncertain-Vol Regime Fits (BTC-Settled)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.show()

### 5.3 Volatility Smile & Backbone

In [ ]:
# Pick the regime with most data for the smile plot
reg = max(fits_regime_mse.keys(), key=lambda r: len(iv1m_df[iv1m_df['regime'] == r]))
g = iv1m_df[iv1m_df['regime'] == reg]
params = fits_regime_mse[reg]['params']

raw_w = params[:3]
weights = np.exp(raw_w) / np.sum(np.exp(raw_w))
betas = params[3:6]
sigmas_ln = np.abs(params[6:9])
S_ref = g['spot'].mean()
sigmas_n = list(np.array(sigmas_ln) * S_ref)

# Choose spot levels within the regime's range
spot_p10 = np.percentile(g['spot'], 10)
spot_p50 = np.percentile(g['spot'], 50)
spot_p90 = np.percentile(g['spot'], 90)
spots_to_plot = [spot_p10, spot_p50, spot_p90]

fig, ax = plt.subplots(figsize=(10, 6))

for S in spots_to_plot:
    K_grid = np.linspace(S * 0.85, S * 1.15, 50)
    smile = [
        op.uncertain_backbone_BTC(S, K, R, TARGET_TAU, weights, betas, sigmas_n, sigmas_ln)
        for K in K_grid
    ]
    ax.plot(K_grid / S, smile, linestyle='--', alpha=0.8, label=f'Smile (S={S:,.0f})')

S_grid = np.linspace(g['spot'].min(), g['spot'].max(), 100)
backbone = [
    op.uncertain_backbone_BTC(S, S, R, TARGET_TAU, weights, betas, sigmas_n, sigmas_ln)
    for S in S_grid
]
ax.plot(S_grid / S_ref, backbone, color='black', linewidth=3, label='Backbone')

ax.set_xlabel('K / S')
ax.set_ylabel('1M Implied Volatility')
ax.set_title(f'BTC Uncertain Volatility Smiles & Backbone ({reg})')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 6. OTM Backbone with Additive Shift

Same methodology as SPX: The uncertain-vol model is fitted only to ATM data.
For OTM strikes, we apply an additive shift:
  IV_OTM(S) = IV_ATM_model(S) + δ_k
where δ_k = mean(OTM IV observed) - mean(ATM model IV)


In [ ]:
def compute_shifted_backbone(k_val, reg, S_grid, weights, betas, sigmas_n, sigmas_ln,
                                     iv1m_k_pc_df, iv1m_df, r, T):
    atm_iv = []
    for S in S_grid:
        try:
            iv = op.uncertain_backbone_BTC(S, S, r, T, weights, betas, sigmas_n, sigmas_ln)
            atm_iv.append(iv if not np.isnan(iv) else np.nan)
        except Exception:
            atm_iv.append(np.nan)

    if k_val == 0:
        return atm_iv, None

    atm_mean = np.nanmean(atm_iv)
    subset = iv1m_k_pc_df[(iv1m_k_pc_df['k'] == k_val) & (iv1m_k_pc_df['regime'] == reg)]
    otm_mean = subset['iv_1m'].mean()
    delta_k = otm_mean - atm_mean

    fitted = [v + delta_k for v in atm_iv]
    return fitted, delta_k

In [ ]:
if os.path.exists(otm_path) and len(iv1m_k_pc_df) > 0:
    unique_k = sorted(iv1m_k_pc_df['k'].unique())
    print(f"Available moneyness levels: {unique_k}")
    
    # Panel 1: -10%, ATM, +10%
    panels_10 = [(-0.10, 'OTM −10% (Put)'), (0.00, 'ATM'), (+0.10, 'OTM +10% (Call)')]
    available_10 = [(k, title) for k, title in panels_10 if k in unique_k]
    
    if len(available_10) == 3:
        all_ivs = []
        for k_val, _ in panels_10:
            subset = iv1m_k_pc_df[iv1m_k_pc_df['k'] == k_val]
            all_ivs.extend(subset['iv_1m'].tolist())
        y_min, y_max = np.percentile(all_ivs, [1, 99])
        y_pad = (y_max - y_min) * 0.1
        
        fig, axes = plt.subplots(1, 3, figsize=(22, 7), sharey=True)
        
        for idx, (k_val, title) in enumerate(panels_10):
            ax = axes[idx]
            subset = iv1m_k_pc_df[iv1m_k_pc_df['k'] == k_val]
            
            if len(subset) > 0:
                for reg_name, g in subset.groupby('regime'):
                    ax.scatter(g['spot'], g['iv_1m'], s=6, alpha=0.35,
                                color=regime_colors.get(reg_name, 'gray'), label=reg_name)
            
            for reg_name, fit_info in fits_regime_mse.items():
                params = fit_info['params']
                raw_w = params[:3]
                weights = np.exp(raw_w) / np.sum(np.exp(raw_w))
                betas = params[3:6]
                sigmas_ln = np.abs(params[6:9])
                
                reg_data = iv1m_df[iv1m_df['regime'] == reg_name]
                if len(reg_data) == 0:
                    continue
                S_ref = reg_data['spot'].mean()
                sigmas_n = list(np.array(sigmas_ln) * S_ref)
                
                S_grid = np.linspace(reg_data['spot'].min(), reg_data['spot'].max(), 150)
                fitted, delta_k = compute_shifted_backbone(
                    k_val, reg_name, S_grid, weights, betas, sigmas_n, sigmas_ln,
                    iv1m_k_pc_df, iv1m_df, R, TARGET_TAU
                )
                
                linestyle = '-' if k_val == 0 else '--'
                label = reg_name if k_val == 0 else f'{reg_name} (δ={delta_k:.4f})' if delta_k is not None else reg_name
                ax.plot(S_grid, fitted, color=regime_colors.get(reg_name, 'gray'),
                        linewidth=2.0, alpha=0.9, linestyle=linestyle)
            
            ax.set_xlabel('BTC Spot Price (USD)', fontsize=11)
            ax.set_ylabel('1M Implied Volatility', fontsize=11)
            shift_note = '' if k_val == 0 else '\n(additive shift applied)'
            ax.set_title(f'{title}{shift_note}', fontsize=13, fontweight='bold')
            ax.set_ylim(y_min - y_pad, y_max + y_pad)
            ax.grid(alpha=0.25)
        
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc='upper center', ncol=4, fontsize=10,
                   bbox_to_anchor=(0.5, 0.98), title='Regime')
        plt.suptitle('BTC Volatility Backbone by Moneyness (BTC-Settled)', fontsize=14, fontweight='bold', y=1.03)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Only {len(available_10)}/3 moneyness levels available for 10% panel")
else:
    print("No OTM data available yet. Run processing pipeline first.")

## 7. Shift Diagnostics

In [ ]:
if os.path.exists(otm_path) and len(iv1m_k_pc_df) > 0:
    shift_rows = []
    
    for reg in REGIMES:
        if reg not in fits_regime_mse:
            continue
        params = fits_regime_mse[reg]['params']
        raw_w = params[:3]
        weights = np.exp(raw_w) / np.sum(np.exp(raw_w))
        betas = params[3:6]
        sigmas_ln = np.abs(params[6:9])
    
        reg_data = iv1m_df[iv1m_df['regime'] == reg]
        S_ref = reg_data['spot'].mean()
        S_grid = np.linspace(reg_data['spot'].min(), reg_data['spot'].max(), 150)
        sigmas_n = list(np.array(sigmas_ln) * S_ref)
    
        atm_iv = [
            op.uncertain_backbone_BTC(S, S, R, TARGET_TAU, weights, betas, sigmas_n, sigmas_ln)
            for S in S_grid
        ]
        atm_mean = np.nanmean(atm_iv)
    
        for k_val in sorted(iv1m_k_pc_df['k'].unique()):
            if k_val == 0:
                shift_rows.append({'regime': reg, 'k': k_val, 'delta_k': 0.0,
                                   'atm_mean': atm_mean, 'otm_mean': atm_mean})
            else:
                subset = iv1m_k_pc_df[(iv1m_k_pc_df['k'] == k_val) & (iv1m_k_pc_df['regime'] == reg)]
                if len(subset) == 0:
                    otm_mean = np.nan
                else:
                    otm_mean = subset['iv_1m'].mean()
                delta_k = otm_mean - atm_mean if not np.isnan(otm_mean) else np.nan
                shift_rows.append({'regime': reg, 'k': k_val, 'delta_k': delta_k,
                                   'atm_mean': atm_mean, 'otm_mean': otm_mean})
    
    shift_df = pd.DataFrame(shift_rows)
    pivot = shift_df.pivot(index='regime', columns='k', values='delta_k')
    print(pivot.to_string(float_format='{:.6f}'.format))
    print()
    print('Additive shifts δ_k = mean(OTM IV) - mean(ATM model IV) by regime and moneyness.')
    print('Positive δ = OTM vol above ATM (typical for puts). Negative δ = below ATM (typical for calls).')
else:
    print("No OTM data available for shift diagnostics.")

## 8. SPX vs BTC Comparison

| Aspect | SPX | BTC |
|--------|-----|-----|
| ATM IV level | 15-50% | 50-150% (expected) |
| Skew shape | Negative smirk | More symmetric smile (expected) |
| Settlement | USD cash-settled | BTC cash-settled (quanto) |
| IV source | Computed from bid/ask mid (standard BS) | Deribit `iv` field (BTC-settled BS) |
| r for fitting | 0.02 | 0.0 |
| β bounds | (0.1, 0.95) | (0.05, 2.0) |
| σ_ln bounds | (0.01, 2.0) | (0.05, 5.0) |
| Active states | 1 (states 1&2 dead) | TBD |


In [ ]:
# Placeholder for SPX vs BTC comparison plots/diagnostics

## 9. Fit Summary

In [ ]:
print("=" * 70)
print("BTC UNCERTAIN-VOL BACKBONE FIT SUMMARY")
print("=" * 70)
print(f"Model: 3-state displaced-diffusion mixture (BTC-settled)")
print(f"Risk-free rate: r = {R}")
print(f"Target maturity: {TARGET_TAU*365:.0f} days")
print(f"Number of ATM observations: {len(iv1m_df)}")
print()

for reg in REGIMES:
    if reg not in fits_regime_mse:
        continue
    info = fits_regime_mse[reg]
    p = info['params']
    w = np.exp(p[:3]) / np.sum(np.exp(p[:3]))
    g = iv1m_df[iv1m_df['regime'] == reg]
    print(f"Regime: {reg}")
    print(f"  N = {len(g)}, MSE = {info['fun']:.6f}, Converged = {info['success']}")
    print(f"  weights   = [{w[0]:.4f}, {w[1]:.4f}, {w[2]:.4f}]")
    print(f"  betas     = [{p[3]:.4f}, {p[4]:.4f}, {p[5]:.4f}]")
    print(f"  sigmas_ln = [{np.abs(p[6]):.4f}, {np.abs(p[7]):.4f}, {np.abs(p[8]):.4f}]")
    S_ref = g['spot'].mean()
    sigmas_n = list(np.array(np.abs(p[6:9])) * S_ref)
    print(f"  S_ref     = {S_ref:.2f}")
    print(f"  sigmas_n  = [{sigmas_n[0]:.2f}, {sigmas_n[1]:.2f}, {sigmas_n[2]:.2f}]")
    
    # Identify dead states (sigma_ln at lower bound)
    for i in range(3):
        status = "DEAD" if np.abs(p[6+i]) < 0.1 else "ACTIVE"
        print(f"  State {i+1}: weight={w[i]:.4f}, beta={p[3+i]:.4f}, "
              f"sigma_ln={np.abs(p[6+i]):.4f}, sigma_n={sigmas_n[i]:.2f} [{status}]")
    print()